<a href="https://colab.research.google.com/github/phong6786789/OMNIVOICE-MOD/blob/main/colab_adam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phong Subi - VOICEOMNI MOD

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [1]:
print('Dang cai dat (~1 phut)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"
print('Cai dat hoan tat!')

# Tai giong mau 10s tu voice-notebooks repo
!wget -q https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples/adam.mp3 -O voice_sample.mp3
print('Da tai voice sample!')

Dang cai dat (~1 phut)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.1 MB/s eta 0:00:00
Cai dat hoan tat!
Da tai voice sample!


In [7]:
print("Đang khởi động - Tạo lại bởi Phong Subi...")

import logging
import os
import re
import time
import numpy as np
import torch
import gradio as gr

# =========================================================
# PATCH TORCH
# =========================================================

import torch as _torch

if not hasattr(_torch, "_utils"):
    _torch._utils = _torch._C._utils


# =========================================================
# PATCH TRANSFORMERS
# =========================================================

import transformers as _tf


class _SafeAutoFeatureExtractor:

    @staticmethod
    def from_pretrained(model_name, **kwargs):

        try:
            from transformers import AutoConfig

            cfg = AutoConfig.from_pretrained(
                model_name,
                trust_remote_code=True,
                **kwargs
            )

            sr = getattr(cfg, "sampling_rate", 24000)

        except Exception:
            sr = 24000

        class _Result:
            sampling_rate = sr

        return _Result()


_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor


# =========================================================
# IMPORT OMNIVOICE
# =========================================================

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device


# =========================================================
# LOGGING
# =========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)


# =========================================================
# KIỂM TRA GPU
# =========================================================

for i in range(30):

    if torch.cuda.is_available():

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

        break

    time.sleep(1)

else:

    print(
        "GPU chưa khả dụng.\n"
        "Hãy chọn: Runtime > Change runtime type > T4 GPU"
    )


# =========================================================
# LOAD MODEL
# =========================================================

DEVICE = get_best_device()

logger.info(
    f"Loading OmniVoice on {DEVICE}..."
)

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=DEVICE,
    dtype=torch.float16,
    load_asr=True
)

SAMPLING_RATE = model.sampling_rate

logger.info(
    f"Model ready — Sampling Rate: {SAMPLING_RATE} Hz"
)


# =========================================================
# DANH SÁCH 26 GIỌNG
# FILE NẰM TRỰC TIẾP TRONG /content/
# =========================================================

VOICE_FILES = {

    # =====================================================
    # GIỌNG NỮ
    # =====================================================

    "♀ Khánh Huyền":
        "/content/khanhhuyentvc_sample.mp3",

    "♀ Thùy Linh":
        "/content/thuylinh_thuyetminh.mp3",

    "♀ Hoài An":
        "/content/vi_female_hoaian_mb.mp3",

    "♀ Hồng Hạnh":
        "/content/vi_female_honghanh_mn_podcast.mp3",

    "♀ Hồng Ngân":
        "/content/vi_female_hongngan_mn_buon.mp3",

    "♀ Khánh Linh":
        "/content/vi_female_khanhlinh_mb.mp3",

    "♀ Kim Phương":
        "/content/vi_female_kimphuong_mb_tKhLy5k.mp3",

    "♀ Nova":
        "/content/vi_female_nova_default.mp3",

    "♀ Thủy Tiên":
        "/content/vi_female_thuytien_mn.mp3",

    "♀ Thùy Trang":
        "/content/vi_female_thuytrang_mb_rzuuQ7F.mp3",

    "♀ Trâm Anh":
        "/content/vi_female_tramanh_mb_sample.mp3",

    "♀ Trần Anh":
        "/content/vi_female_trananh_mb.mp3",


    # =====================================================
    # GIỌNG NAM
    # =====================================================

    "♂ Đức Trọng":
        "/content/ductrong_sample2.mp3",

    "♂ Đăng Khoa":
        "/content/vi_male_dangkhoa_mb.mp3",

    "♂ Echo":
        "/content/vi_male_echo_default.mp3",

    "♂ Lê Đức":
        "/content/vi_male_leduc_mb_FZBJAUZ.mp3",

    "♂ Lê Hoàng":
        "/content/vi_male_lehoang_mb_SRojRwi.mp3",

    "♂ Lê Nghĩa":
        "/content/vi_male_lenghia_mb_BITWyO7.mp3",

    "♂ Minh Quân":
        "/content/vi_male_minhquan_mb.mp3",

    "♂ Minh Triết":
        "/content/vi_male_minhtriet_mb.mp3",

    "♂ Onyx":
        "/content/vi_male_onyx_default.mp3",

    "♂ Thành Trung":
        "/content/vi_male_thanhtrung_mn_k8K8ONG.mp3",

    "♂ Trí Dũng":
        "/content/vi_male_tridung_mn_sample.mp3",

    "♂ Tuấn Kiệt":
        "/content/vi_male_tuankiet_mn.mp3",

    "♂ Văn Đức":
        "/content/vi_male_vanduc_mn.mp3",

    "♂ Văn Duy":
        "/content/vi_male_vanduy_mb.mp3",
}


# =========================================================
# KIỂM TRA FILE GIỌNG
# =========================================================

print("\n========================================")
print("KIỂM TRA 26 GIỌNG")
print("========================================")

available_voices = []
missing_voices = []

for voice_name, path in VOICE_FILES.items():

    if os.path.exists(path):

        print(
            f"✅ {voice_name}"
        )

        available_voices.append(
            voice_name
        )

    else:

        print(
            f"❌ {voice_name}"
        )

        print(
            f"   Không tìm thấy: {path}"
        )

        missing_voices.append(
            voice_name
        )


print("\n========================================")

print(
    f"Sẵn sàng: {len(available_voices)}/26 giọng"
)

print(
    f"Thiếu: {len(missing_voices)} giọng"
)

print("========================================\n")


# =========================================================
# CACHE VOICE PROMPT
# MỖI GIỌNG CHỈ TẠO PROMPT 1 LẦN
# =========================================================

VOICE_PROMPT_CACHE = {}


def get_voice_prompt(voice_name):

    """
    Tạo voice clone prompt lần đầu.
    Những lần sau lấy từ cache.
    """

    if voice_name in VOICE_PROMPT_CACHE:

        logger.info(
            f"Using cached voice: {voice_name}"
        )

        return VOICE_PROMPT_CACHE[
            voice_name
        ]


    if voice_name not in VOICE_FILES:

        raise ValueError(
            "Không tìm thấy giọng đã chọn."
        )


    voice_file = VOICE_FILES[
        voice_name
    ]


    if not os.path.exists(
        voice_file
    ):

        raise FileNotFoundError(
            f"Không tìm thấy file:\n{voice_file}"
        )


    logger.info(
        f"Creating VoiceClonePrompt: {voice_name}"
    )


    voice_prompt = model.create_voice_clone_prompt(
        ref_audio=voice_file
    )


    VOICE_PROMPT_CACHE[
        voice_name
    ] = voice_prompt


    logger.info(
        f"Voice cached: {voice_name}"
    )


    return voice_prompt


# =========================================================
# OMNIVOICE GENERATION CONFIG
# =========================================================

GEN_CFG = OmniVoiceGenerationConfig(

    num_step=32,

    guidance_scale=1.8,

    denoise=True,

    preprocess_prompt=True,

    postprocess_output=True,

    position_temperature=5.0,

    class_temperature=0.2,

    pad_duration=0.1,

    fade_duration=0.1,
)


# =========================================================
# NGHE THỬ GIỌNG
# =========================================================

def preview_voice(
    voice_name
):

    """
    Khi đổi dropdown,
    trả về file MP3 mẫu tương ứng.
    """

    if not voice_name:

        return None


    path = VOICE_FILES.get(
        voice_name
    )


    if not path:

        return None


    if not os.path.exists(
        path
    ):

        raise gr.Error(
            f"Không tìm thấy file nghe thử:\n{path}"
        )


    return path


# =========================================================
# CHUẨN HÓA AUDIO
# =========================================================

def normalize_audio(
    audio
):

    """
    Chuyển Tensor / array về numpy float32
    và chống clipping.
    """

    if torch.is_tensor(
        audio
    ):

        audio = (
            audio
            .detach()
            .float()
            .cpu()
            .numpy()
        )


    audio = np.asarray(
        audio,
        dtype=np.float32
    )


    audio = np.squeeze(
        audio
    )


    max_amp = np.max(
        np.abs(audio)
    )


    if max_amp > 1.0:

        audio = (
            audio / max_amp
        )


    return audio


# =========================================================
# TẠO GIỌNG NÓI
# =========================================================

def generate_voice(
    text,
    voice_name,
    speed,
    pause_duration
):

    # -----------------------------------------------------
    # CHECK TEXT
    # -----------------------------------------------------

    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    text = text.strip()


    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    # -----------------------------------------------------
    # CHECK VOICE
    # -----------------------------------------------------

    if voice_name not in VOICE_FILES:

        raise gr.Error(
            "Giọng đã chọn không hợp lệ."
        )


    voice_file = VOICE_FILES[
        voice_name
    ]


    if not os.path.exists(
        voice_file
    ):

        raise gr.Error(
            f"Không tìm thấy file giọng:\n{voice_file}"
        )


    # -----------------------------------------------------
    # GET/CACHE VOICE PROMPT
    # -----------------------------------------------------

    try:

        voice_prompt = get_voice_prompt(
            voice_name
        )

    except Exception as e:

        raise gr.Error(
            f"Lỗi tạo VoiceClonePrompt:\n{e}"
        )


    # -----------------------------------------------------
    # CHIA ĐOẠN
    # Hai lần xuống dòng = đoạn mới
    # -----------------------------------------------------

    paragraphs = [

        p.strip()

        for p in re.split(
            r"\n\s*\n",
            text
        )

        if p.strip()

    ]


    if not paragraphs:

        raise gr.Error(
            "Không có nội dung hợp lệ."
        )


    logger.info(
        f"Voice: {voice_name}"
    )

    logger.info(
        f"Paragraphs: {len(paragraphs)}"
    )


    # -----------------------------------------------------
    # GENERATE
    # -----------------------------------------------------

    all_audio = []


    for i, paragraph in enumerate(
        paragraphs
    ):

        logger.info(
            f"Generating paragraph "
            f"{i + 1}/{len(paragraphs)}"
        )


        try:

            result = model.generate(

                text=paragraph,

                voice_clone_prompt=voice_prompt,

                language="vi",

                speed=float(
                    speed
                ),

                generation_config=GEN_CFG

            )

        except Exception as e:

            raise gr.Error(
                f"Lỗi tạo audio ở đoạn {i + 1}:\n{e}"
            )


        audio = result[0]


        audio = normalize_audio(
            audio
        )


        all_audio.append(
            audio
        )


        # -------------------------------------------------
        # THÊM SILENCE GIỮA CÁC ĐOẠN
        # -------------------------------------------------

        if i < len(paragraphs) - 1:

            silence_samples = int(

                SAMPLING_RATE
                * float(
                    pause_duration
                )

            )


            silence = np.zeros(

                silence_samples,

                dtype=np.float32

            )


            all_audio.append(
                silence
            )


    # -----------------------------------------------------
    # NỐI AUDIO
    # -----------------------------------------------------

    final_audio = np.concatenate(
        all_audio
    )


    final_audio = normalize_audio(
        final_audio
    )


    # -----------------------------------------------------
    # FLOAT32 -> INT16
    # -----------------------------------------------------

    waveform = (

        final_audio
        * 32767

    ).astype(
        np.int16
    )


    return (

        SAMPLING_RATE,

        waveform

    )


# =========================================================
# CSS - GIỮ GIAO DIỆN ỔN ĐỊNH
# =========================================================

CSS = """

/* ========================================
   CONTAINER
======================================== */

.gradio-container {

    max-width: 820px !important;

    width: 100% !important;

    margin: 0 auto !important;

    padding: 18px !important;

    box-sizing: border-box !important;

}


/* ========================================
   TẤT CẢ ELEMENT
======================================== */

.gradio-container * {

    box-sizing: border-box !important;

}


/* ========================================
   CHỐNG NHẢY NGANG
======================================== */

html {

    overflow-y: scroll !important;

}

body {

    overflow-x: hidden !important;

}


/* ========================================
   TEXTBOX
======================================== */

textarea {

    resize: vertical !important;

    min-height: 200px !important;

}


/* ========================================
   DROPDOWN / INPUT
======================================== */

.gradio-container input,

.gradio-container textarea,

.gradio-container select {

    width: 100% !important;

}


/* ========================================
   AUDIO PLAYER
======================================== */

[data-testid="audio"] {

    min-height: 90px !important;

}


/* ========================================
   BUTTON
======================================== */

button {

    min-height: 46px !important;

}


/* ========================================
   FOOTER
======================================== */

footer {

    display: none !important;

}

"""


# =========================================================
# THEME
# =========================================================

THEME = gr.themes.Soft(
    primary_hue="indigo"
)


# =========================================================
# GIAO DIỆN
# =========================================================

print(
    "Khởi động giao diện Kiên Đoàn TTS..."
)


gr.close_all()


# Nếu Onyx bị thiếu thì lấy giọng đầu tiên có sẵn
DEFAULT_VOICE = "♂ Onyx"

if DEFAULT_VOICE not in available_voices:

    if available_voices:

        DEFAULT_VOICE = available_voices[0]

    else:

        DEFAULT_VOICE = list(
            VOICE_FILES.keys()
        )[0]


with gr.Blocks(

    title="Kiên Đoàn TTS — 26 Voices",

    theme=THEME,

    css=CSS,

    fill_width=True

) as demo:


    # =====================================================
    # TITLE
    # =====================================================

    gr.Markdown(
        """
# 🎙️ Kiên Đoàn TTS

### 26 giọng tiếng Việt

Chọn giọng → nghe thử → nhập nội dung → tạo giọng nói.
"""
    )


    # =====================================================
    # CHỌN GIỌNG
    # =====================================================

    voice_selector = gr.Dropdown(

        choices=list(
            VOICE_FILES.keys()
        ),

        value=DEFAULT_VOICE,

        label="🎤 Chọn giọng",

        interactive=True

    )


    # =====================================================
    # NGHE THỬ
    # =====================================================

    default_preview = VOICE_FILES.get(
        DEFAULT_VOICE
    )


    if not os.path.exists(
        default_preview
    ):

        default_preview = None


    preview_audio = gr.Audio(

        value=default_preview,

        label="🔊 Nghe thử giọng",

        interactive=False,

        autoplay=False

    )


    # =====================================================
    # NỘI DUNG
    # =====================================================

    text_input = gr.Textbox(

        label="📝 Nội dung",

        lines=10,

        placeholder=(
            "Nhập văn bản bạn muốn chuyển thành giọng nói...\n\n"
            "Muốn nghỉ lâu hơn giữa các đoạn thì xuống dòng 2 lần."
        )

    )


    # =====================================================
    # SPEED
    # =====================================================

    speed_slider = gr.Slider(

        minimum=0.70,

        maximum=1.30,

        value=0.95,

        step=0.05,

        label="⚡ Tốc độ đọc"

    )


    # =====================================================
    # PAUSE
    # =====================================================

    pause_slider = gr.Slider(

        minimum=0,

        maximum=2.0,

        value=0.3,

        step=0.1,

        label="⏸ Nghỉ giữa các đoạn (giây)"

    )


    # =====================================================
    # GENERATE BUTTON
    # =====================================================

    generate_button = gr.Button(

        "🎙️ TẠO GIỌNG NÓI",

        variant="primary"

    )


    # =====================================================
    # OUTPUT AUDIO
    # =====================================================

    output_audio = gr.Audio(

        label="🎧 Kết quả",

        autoplay=False

    )


    # =====================================================
    # EVENTS
    # =====================================================

    voice_selector.change(

        fn=preview_voice,

        inputs=voice_selector,

        outputs=preview_audio,

        show_progress="hidden"

    )


    generate_button.click(

        fn=generate_voice,

        inputs=[

            text_input,

            voice_selector,

            speed_slider,

            pause_slider

        ],

        outputs=output_audio,

        concurrency_limit=1,

        show_progress="minimal"

    )


# =========================================================
# QUEUE
# =========================================================

demo.queue(
    default_concurrency_limit=1
)


# =========================================================
# LAUNCH
# =========================================================

demo.launch(

    server_name="0.0.0.0",

    share=True,

    show_error=True

)

Đang khởi động - Tạo lại bởi Phong Subi...
GPU: Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

...

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]


KIỂM TRA 26 GIỌNG
✅ ♀ Khánh Huyền
✅ ♀ Thùy Linh
✅ ♀ Hoài An
✅ ♀ Hồng Hạnh
✅ ♀ Hồng Ngân
✅ ♀ Khánh Linh
✅ ♀ Kim Phương
✅ ♀ Nova
✅ ♀ Thủy Tiên
✅ ♀ Thùy Trang
✅ ♀ Trâm Anh
✅ ♀ Trần Anh
✅ ♂ Đức Trọng
✅ ♂ Đăng Khoa
✅ ♂ Echo
✅ ♂ Lê Đức
✅ ♂ Lê Hoàng
✅ ♂ Lê Nghĩa
✅ ♂ Minh Quân
✅ ♂ Minh Triết
✅ ♂ Onyx
✅ ♂ Thành Trung
✅ ♂ Trí Dũng
✅ ♂ Tuấn Kiệt
✅ ♂ Văn Đức
✅ ♂ Văn Duy

Sẵn sàng: 26/26 giọng
Thiếu: 0 giọng

Khởi động giao diện Kiên Đoàn TTS...
Closing server running on port: 7860


/tmp/ipykernel_961/3544539092.py:844: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://79f82698ece5505fa6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
